In [ ]:

import toytree, toyplot, toyplot.png
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import glob, os, re
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load all segment trees
tree_dir = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/iqtree"
trees = {}
for f in sorted(glob.glob(os.path.join(tree_dir, "segment_*.treefile"))):
    seg = re.search(r'segment_(\d+)', os.path.basename(f)).group(0)
    trees[seg] = toytree.tree(f)
    print(f"{seg}: {trees[seg].ntips} tips")

In [ ]:
COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990",
    "#dcbeff", "#9A6324", "#800000", "#aaffc3", "#000075",
    "#ffe119", "#808000", "#ffd8b1", "#a9a9a9", "#000000",
]

def cluster_tree(tre, n_clusters):
    """Cluster tips by patristic distance, return DataFrame."""
    tips = tre.get_tip_labels()
    dmat = tre.distance.get_tip_distance_matrix()
    condensed = squareform(dmat, checks=False)
    Z = linkage(condensed, method="average")
    labels = fcluster(Z, t=n_clusters, criterion="maxclust")
    return pd.DataFrame({"tip_label": tips, "cluster": labels}).sort_values("cluster").reset_index(drop=True)

In [ ]:
seg_dd = widgets.Dropdown(options=list(trees.keys()), description="Segment:")
k_slider = widgets.IntSlider(value=4, min=2, max=20, description="Clusters:")
plot_toggle = widgets.Checkbox(value=False, description="Draw tree (slow if >1000 tips)")
run_btn = widgets.Button(description="Run", button_style="success")
out = widgets.Output()

cluster_dfs = {}  # stores result: {cluster_id: DataFrame}

def on_run(b):
    with out:
        clear_output(wait=True)
        seg, k = seg_dd.value, k_slider.value
        tre = trees[seg]

        # Cluster
        df = cluster_tree(tre, k)

        # Optionally draw
        if plot_toggle.value:
            # Build color arrays in node-index order
            tip_colors = [COLORS[(c - 1) % len(COLORS)] for c in df["cluster"]]
            # Reorder to match tree tip idx order (0..ntips-1)
            tip_order = tre.get_tip_labels()
            color_map = dict(zip(df["tip_label"], tip_colors))
            ordered_tip_colors = [color_map[t] for t in tip_order]

            # Edge colors: color edge if all descendant tips share a cluster
            cluster_map = dict(zip(df["tip_label"], df["cluster"]))

            canvas, axes, mark = tre.draw(
                width=1000,
                height=max(500, tre.ntips * 8),
                tip_labels_style={"font-size": "5px"},
                tip_labels_align=True,
                node_sizes=0,
                tip_labels_colors=ordered_tip_colors,
                edge_widths=1,
            )
            display(canvas)

        # Show per-cluster DataFrames
        cluster_dfs.clear()
        for cid in sorted(df["cluster"].unique()):
            sub = df[df["cluster"] == cid].reset_index(drop=True)
            cluster_dfs[cid] = sub
            print(f"\n── Cluster {cid} ({len(sub)} tips) ──")
            display(sub)

run_btn.on_click(on_run)
display(widgets.VBox([widgets.HBox([seg_dd, k_slider, plot_toggle, run_btn]), out]))

In [ ]:
# After clicking Run, cluster_dfs is a dict: {1: DataFrame, 2: DataFrame, ...}
# Example:
for cid, cdf in cluster_dfs.items():
    print(f"Cluster {cid}: {len(cdf)} tips")

# Export all to CSV:
# pd.concat(cluster_dfs.values()).to_csv("/tmp/clusters.csv", index=False)